In [12]:
%cd /content/data

/content/data


In [16]:
# Fix the remaining issues with drivers.txt and circuits.json
import pandas as pd
import json
import io

# Fix drivers.txt - try different separators and handle UTF-16 properly
print("=== FIXING DRIVERS.TXT ===")
if 'drivers.txt' in uploaded:
    raw_data = uploaded['drivers.txt']

    # Try different combinations of encoding and separators
    drivers_configs = [
        {'encoding': 'utf-16', 'sep': '\t'},  # UTF-16 with tabs
        {'encoding': 'utf-16', 'sep': ','},   # UTF-16 with commas
        {'encoding': 'utf-8', 'sep': '\t'},   # UTF-8 with tabs
        {'encoding': 'latin1', 'sep': '\t'},  # Latin1 with tabs
        {'encoding': 'cp1252', 'sep': '\t'},  # CP1252 with tabs
    ]

    df_drivers = None
    for config in drivers_configs:
        try:
            print(f"Trying: {config}")
            df_drivers = pd.read_csv(
                io.BytesIO(raw_data),
                encoding=config['encoding'],
                sep=config['sep'],
                na_values=['\\N'],
                on_bad_lines='skip'  # Skip problematic lines
            )
            print(f"SUCCESS with {config}")
            print(f"Shape: {df_drivers.shape}")
            print(f"Columns: {df_drivers.columns.tolist()}")
            break
        except Exception as e:
            print(f"Failed: {e}")
            continue

    if df_drivers is not None:
        print("\n[drivers.txt] Successfully loaded!")
        print(df_drivers.head())

        # Clean up the data
        if 'driverId' in df_drivers.columns:
            df_drivers['driverId'] = pd.to_numeric(df_drivers['driverId'], errors='coerce').astype('Int64')

        # Store in cleaned dataframes
        cleaned_dataframes['drivers'] = df_drivers
        globals()['df_drivers'] = df_drivers
    else:
        print("[drivers.txt] All attempts failed - manual inspection needed")

print("\n=== FIXING CIRCUITS.JSON ===")
if 'circuits.json' in uploaded:
    try:
        # Load raw JSON and inspect structure
        json_text = uploaded['circuits.json'].decode('utf-8')
        json_data = json.loads(json_text)

        print(f"JSON type: {type(json_data)}")
        print(f"JSON keys: {list(json_data.keys())}")

        # The issue: JSON is structured as {field: [values]} instead of [{record1}, {record2}, ...]
        # Need to reconstruct it properly

        if isinstance(json_data, dict) and 'circuitId' in json_data:
            # Extract all the arrays
            circuit_ids = json_data.get('circuitId', [])
            circuit_refs = json_data.get('circuitRef', [])
            names = json_data.get('name', [])
            locations = json_data.get('location', [])
            countries = json_data.get('country', [])
            lats = json_data.get('lat', [])
            lngs = json_data.get('lng', [])
            alts = json_data.get('alt', [])
            urls = json_data.get('url', [])

            # Get the length (should all be the same)
            n_circuits = len(circuit_ids)
            print(f"Number of circuits: {n_circuits}")

            # Reconstruct as proper records
            circuits_records = []
            for i in range(n_circuits):
                record = {
                    'circuitId': circuit_ids[i] if i < len(circuit_ids) else None,
                    'circuitRef': circuit_refs[i] if i < len(circuit_refs) else None,
                    'name': names[i] if i < len(names) else None,
                    'location': locations[i] if i < len(locations) else None,
                    'country': countries[i] if i < len(countries) else None,
                    'lat': lats[i] if i < len(lats) else None,
                    'lng': lngs[i] if i < len(lngs) else None,
                    'alt': alts[i] if i < len(alts) else None,
                    'url': urls[i] if i < len(urls) else None,
                }
                circuits_records.append(record)

            # Create DataFrame from records
            df_circuits = pd.DataFrame(circuits_records)

            print(f"\n[circuits.json] Reconstructed successfully!")
            print(f"Shape: {df_circuits.shape}")
            print(f"Columns: {df_circuits.columns.tolist()}")
            print("\nFirst few rows:")
            print(df_circuits.head())

            # Clean up data types
            df_circuits['circuitId'] = pd.to_numeric(df_circuits['circuitId'], errors='coerce').astype('Int64')
            df_circuits['lat'] = pd.to_numeric(df_circuits['lat'], errors='coerce')
            df_circuits['lng'] = pd.to_numeric(df_circuits['lng'], errors='coerce')
            df_circuits['alt'] = pd.to_numeric(df_circuits['alt'], errors='coerce').astype('Int64')

            # Store in cleaned dataframes
            cleaned_dataframes['circuits'] = df_circuits
            globals()['df_circuits'] = df_circuits

        else:
            print("Unexpected JSON structure - manual handling needed")
            print(f"First few keys/values: {dict(list(json_data.items())[:3])}")

    except Exception as e:
        print(f"Error fixing circuits.json: {e}")

# Final status report
print("\n" + "="*50)
print("  FINAL DATA CLEANING STATUS  ")
print("="*50)

final_datasets = {}
for key, df in cleaned_dataframes.items():
    final_datasets[key] = df.shape
    print(f"✅ df_{key}: {df.shape[0]:,} rows × {df.shape[1]} columns")

missing_datasets = []
expected_datasets = ['results', 'races', 'constructors', 'drivers', 'circuits']
for dataset in expected_datasets:
    if dataset not in cleaned_dataframes:
        missing_datasets.append(dataset)
        print(f"❌ df_{dataset}: FAILED TO LOAD")

print(f"\nSUCCESS RATE: {len(cleaned_dataframes)}/{len(expected_datasets)} datasets loaded")

if len(cleaned_dataframes) == len(expected_datasets):
    print("🎉 ALL DATASETS SUCCESSFULLY CLEANED!")
    print("\nYou can now proceed with your F1 analysis using:")
    for dataset in expected_datasets:
        print(f"   df_{dataset}.head()")
else:
    print(f"\n⚠️  {len(missing_datasets)} dataset(s) still need manual fixing:")
    for dataset in missing_datasets:
        print(f"   - {dataset}")

print("\n" + "="*50)

=== FIXING DRIVERS.TXT ===
Trying: {'encoding': 'utf-16', 'sep': '\t'}
SUCCESS with {'encoding': 'utf-16', 'sep': '\t'}
Shape: (854, 9)
Columns: ['driverId', 'driverRef', 'number', 'code', 'forename', 'surname', 'dob', 'nationality', 'url']

[drivers.txt] Successfully loaded!
   driverId   driverRef  number code  forename     surname         dob  \
0         1    hamilton    44.0  HAM     Lewis    Hamilton  07/01/1985   
1         2    heidfeld     NaN  HEI      Nick    Heidfeld  10/05/1977   
2         3     rosberg     6.0  ROS      Nico     Rosberg  27/06/1985   
3         4      alonso    14.0  ALO  Fernando      Alonso  29/07/1981   
4         5  kovalainen     NaN  KOV    Heikki  Kovalainen  19/10/1981   

  nationality                                             url  
0     British     http://en.wikipedia.org/wiki/Lewis_Hamilton  
1      German      http://en.wikipedia.org/wiki/Nick_Heidfeld  
2      German       http://en.wikipedia.org/wiki/Nico_Rosberg  
3     Spanish    http:

In [6]:
%cd /content/data

/content/data
